# 활동 9 컴퓨터 비전 객체 탐지 기술 체험하기

교과서 145쪽

**YOLO**(You Only Look Once)는 사진을 한 번만 보고 그 안의 여러 물체가 **어디에(위치) 무엇이(종류)** 있는지 동시에 찾는 객체 탐지 모델입니다. 이 노트북은 YOLO의 최신 버전을 이용해 교과서 활동 9를 더 빠르게 해 볼 수 있게 만든 것입니다.

**사용 방법**
- 위에서부터 차례로 각 칸 왼쪽의 ▶ 버튼을 눌러 실행합니다. 오른쪽의 선택 상자와 슬라이더로 설정을 바꿉니다.
- ①은 처음 한 번만 실행하면 됩니다(1분 정도 걸립니다).
- 내 사진을 올려도 사진은 내 코랩 세션에서만 쓰이고 수업이 끝나면 지워집니다.

In [ ]:
#@title ① 준비하기: YOLO 설치하고 사전 훈련된 모델 불러오기
!pip -q install ultralytics
from ultralytics import YOLO
import matplotlib.pyplot as plt
from collections import Counter

# COCO 데이터셋(사진 약 12만 장, 물체 80종류)으로 미리 학습한 가장 작은 YOLO 모델
model = YOLO('yolo11n.pt')
KO = {"person": "사람", "bicycle": "자전거", "car": "자동차", "motorcycle": "오토바이", "airplane": "비행기", "bus": "버스", "train": "기차", "truck": "트럭", "boat": "배", "traffic light": "신호등", "fire hydrant": "소화전", "stop sign": "정지 표지판", "parking meter": "주차 미터기", "bench": "벤치", "bird": "새", "cat": "고양이", "dog": "개", "horse": "말", "sheep": "양", "cow": "소", "elephant": "코끼리", "bear": "곰", "zebra": "얼룩말", "giraffe": "기린", "backpack": "배낭", "umbrella": "우산", "handbag": "핸드백", "tie": "넥타이", "suitcase": "여행 가방", "frisbee": "원반", "skis": "스키", "snowboard": "스노보드", "sports ball": "공", "kite": "연", "baseball bat": "야구 방망이", "baseball glove": "야구 글러브", "skateboard": "스케이트보드", "surfboard": "서핑보드", "tennis racket": "테니스 라켓", "bottle": "병", "wine glass": "와인잔", "cup": "컵", "fork": "포크", "knife": "칼", "spoon": "숟가락", "bowl": "그릇", "banana": "바나나", "apple": "사과", "sandwich": "샌드위치", "orange": "오렌지", "broccoli": "브로콜리", "carrot": "당근", "hot dog": "핫도그", "pizza": "피자", "donut": "도넛", "cake": "케이크", "chair": "의자", "couch": "소파", "potted plant": "화분", "bed": "침대", "dining table": "식탁", "toilet": "변기", "tv": "텔레비전", "laptop": "노트북", "mouse": "마우스", "remote": "리모컨", "keyboard": "키보드", "cell phone": "휴대폰", "microwave": "전자레인지", "oven": "오븐", "toaster": "토스터", "sink": "싱크대", "refrigerator": "냉장고", "book": "책", "clock": "시계", "vase": "꽃병", "scissors": "가위", "teddy bear": "곰 인형", "hair drier": "헤어드라이어", "toothbrush": "칫솔"}
print('준비 완료! 이 모델이 찾을 수 있는 물체:', len(model.names), '종류')
print(', '.join(KO.get(n, n) for n in list(model.names.values())[:20]), '...')

In [ ]:
#@title ② 사진 고르기
photo = "예시 사진 1: 거리의 버스" #@param ["예시 사진 1: 거리의 버스", "예시 사진 2: 사람들", "교과서 그림 (test.jpg)", "내 사진 올리기"]

if photo.startswith('예시 사진 1'):
    source = 'https://ultralytics.com/images/bus.jpg'
elif photo.startswith('예시 사진 2'):
    source = 'https://ultralytics.com/images/zidane.jpg'
elif photo.startswith('교과서'):
    import urllib.request
    urllib.request.urlretrieve('https://aibook-dl.vercel.app/data/test.jpg', 'test.jpg')
    source = 'test.jpg'
else:
    from google.colab import files
    up = files.upload()          # 버튼을 눌러 사진 파일을 고르세요
    source = list(up.keys())[0]
print('고른 사진:', source)

In [ ]:
#@title ③ 객체 탐지하기: 신뢰도 기준을 바꿔 다시 실행해 보세요
confidence = 0.25 #@param {type:"slider", min:0.05, max:0.95, step:0.05}

result = model.predict(source, conf=confidence, verbose=False)[0]
plt.figure(figsize=(9, 7))
plt.imshow(result.plot()[:, :, ::-1])    # 상자와 이름, 신뢰도를 그린 사진
plt.axis('off')
plt.show()

names = [model.names[int(c)] for c in result.boxes.cls]
scores = [float(s) for s in result.boxes.conf]
print(f'신뢰도 {confidence} 이상인 물체 {len(names)}개를 찾았습니다.')
for name, n in Counter(names).most_common():
    print(f'  - {KO.get(name, name)}({name}): {n}개')
print()
print('찾은 물체마다의 신뢰도')
for name, s in sorted(zip(names, scores), key=lambda t: -t[1]):
    print(f'  {KO.get(name, name):8s} {s:.2f}')

### 생각해 보기
1. 신뢰도 기준(confidence)을 0.05로 낮추면 무엇이 달라지나요? 0.9로 높이면요?
2. 모델이 찾지 못한 물체나 잘못 찾은 물체가 있나요? 왜 그랬을까요?
3. 객체 탐지 기술을 학교나 우리 동네에서 어디에 쓸 수 있을까요?